# 01 · Do the new BRIGHT instance labels cover Ukraine, Mexico or Myanmar?

**Task 4 of the next-steps guide.** BRIGHT's Zenodo record was updated on 25 Mar 2026 with
`cvprw26_trainval_instance_labels.zip` (31 MB). If it covers Ukraine, the Ukraine fallback chain is not needed.

**Rule:** for confirmatory events we only look at **file names and file counts**, never at the damage
classes inside the files. The structure of the files is inspected on Beirut (a development event) only.

In [ ]:
import os, re, zipfile, urllib.request, collections, json
URL = "https://zenodo.org/records/19216381/files/cvprw26_trainval_instance_labels.zip?download=1"
ZIP = "/content/cvprw26_trainval_instance_labels.zip"
if not os.path.exists(ZIP):
    urllib.request.urlretrieve(URL, ZIP)
print(os.path.getsize(ZIP) / 1e6, "MB")

with zipfile.ZipFile(ZIP) as z:
    names = [n for n in z.namelist() if not n.endswith("/")]
print(len(names), "files; first 10:")
for n in names[:10]:
    print("  ", n)
print("extensions:", collections.Counter(os.path.splitext(n)[1].lower() for n in names))

In [ ]:
# Files per event (event = part of the file name before the first "-")
EVENTS = {
    "confirmatory": ["haiti", "turkey", "morocco", "noto", "bata", "ukraine"],
    "development": ["beirut", "la_palma", "hawaii", "congo", "libya"],
    "secondary": ["marshall", "mexico", "myanmar"],
}
def event_of(path):
    base = os.path.basename(path)
    return base.split("-", 1)[0].lower() if "-" in base else "?"

per_event = collections.Counter(event_of(n) for n in names)

# Fallback: labels shipped as one or a few COCO-style JSON files ("images" list with file names).
# Only the image *file names* are counted, so no confirmatory damage class is read.
known = set(sum(EVENTS.values(), []))
if sum(v for k, v in per_event.items() if k in known) == 0:
    per_event = collections.Counter()
    with zipfile.ZipFile(ZIP) as z:
        for n in names:
            if n.lower().endswith(".json"):
                obj = json.loads(z.read(n))
                if isinstance(obj, dict) and isinstance(obj.get("images"), list):
                    for im in obj["images"]:
                        per_event[event_of(str(im.get("file_name", "")))] += 1
                    print(f"{n}: COCO-style, {len(obj['images'])} images")
print(f"{'role':<13} {'event':<9} files")
for role, evs in EVENTS.items():
    for e in evs:
        print(f"{role:<13} {e:<9} {per_event.get(e, 0)}")
others = {k: v for k, v in per_event.items() if k not in sum(EVENTS.values(), [])}
print("other names:", others)
print()
for e in ["ukraine", "mexico", "myanmar"]:
    print(f"{e}: {'COVERED' if per_event.get(e, 0) else 'not covered'} ({per_event.get(e, 0)} files)")

In [ ]:
# Structure of one Beirut file only (development event: allowed)
beirut = sorted(n for n in names if event_of(n) == "beirut")
print(len(beirut), "Beirut files; inspecting", beirut[0] if beirut else None)
if beirut:
    with zipfile.ZipFile(ZIP) as z:
        raw = z.read(beirut[0])
    if beirut[0].lower().endswith(".json"):
        obj = json.loads(raw)
        def show(o, depth=0, maxd=3):
            pad = "  " * depth
            if isinstance(o, dict):
                for k, v in list(o.items())[:12]:
                    print(f"{pad}{k}: {type(v).__name__}")
                    if depth < maxd: show(v, depth + 1)
            elif isinstance(o, list) and o:
                print(f"{pad}[list of {len(o)}] first item:")
                if depth < maxd: show(o[0], depth + 1)
        show(obj)
    else:
        print("not JSON; first bytes:", raw[:16])

**Record the result** as one line in `EXPERIMENT_LOG.md`, e.g.
`2026-11-xx | Checked cvprw26 instance labels | — | Zenodo 19216381 | 0 | Ukraine: covered / not covered (n files); Mexico: …; Myanmar: … | names only for confirmatory`.

If Ukraine is covered, update `configs/events.yaml` (`ukraine: units: instance`) and §2 of `docs/preregistration.md`.